# Pivot Tables and Data Wrangling — Solutions Walkthrough (Task 22)

This is the fully worked companion to the Task 22 practice set. Where the practice notebook posed the questions, this one shows the reasoning and the code behind each answer, so you can compare your approach against a reference implementation and understand *why* it works. The unifying thread is the **pivot table**, but no question is solved by that call alone — each demands cleaning, feature engineering, parsing, joining, and plotting as well.

**Q-1** builds a pivot table on a large NFT marketplace dump (~310,000 rows, 26 columns). Using `verification_status`, `contract_type`, `rarity_score`, and `last_sale_price`, it computes summed totals — and because `values` is a *list* of two columns, the result naturally carries a two-level column header.

**Q-2** is the dirty-data pivot: the car-price dataset has no `Brand` column (it must be split out of the `Name` field) and a `Year` column containing malformed values such as `20173` (repaired with a regular-expression extraction of the four-digit year). The output is a brand × year grid mixing **mean Price** and **median Miles** — two different aggregations — visualised with KDE plots for each.

**Q-3** tackles India's daily power-generation data. A `required` column is derived from `Actual(MU)` and `Excess(+)/Shortfall(-)` so supply exactly meets demand; the month is extracted with `dt.month_name()`; the top-10 most frequent power stations are selected; and a month × station pivot of summed required generation is plotted as a line chart.

**Q-4 and Q-5** parse the `question-answer.csv` file. Questions and answers are identified by their `Q<n>`/`Ans<n>` prefixes, split off with `.str.startswith` and `.str.split('-').str.get(1)`, and stitched into a two-column DataFrame via an index-merge. Q-5 then filters to questions containing no `?`, using an escaped regex.

**Q6–Q10** join the `log_file` and `employee` tables on the mismatched keys `emp_id`/`EMPLOYEE_ID` to answer activity questions by month, weekday, and weekend. **Q-11** repairs the employee table: reconstructing missing first names from the email stem, appending `@campusx.com`, and filtering full names that start with 'A' and performed specific activities. Every step's output is printed, so you can verify each intermediate result.

## 0. Setup

Import `pandas`, `numpy`, and `re`, and show all columns with `pd.options.display.max_columns = None`.

In [1]:
import pandas as pd
import numpy as np
import re

pd.options.display.max_columns = None

# 1. NFT Pivot Table (Q-1)

A large NFT sales dataset, pivoted to show total sale values and rarity scores across verification and contract types.

### `Question-1:`

The dataset is based on NFTs. This is quite large. So, when you will read the dataset, you have to wait. The link of the dataset: https://tinyurl.com/2pfhncqw

Your task is to make a pivot table by using the columns "verification_status", "contract_type", "rarity_score" and "last_sale_price" to find out the total values.

## 1.1 Loading a large dataset

`pd.read_csv('https://tinyurl.com/2pfhncqw')` loads the NFT marketplace data, and the notebook warns that it is large — `df.shape` confirms **(309910, 26)**, some 310,000 rows across 26 columns. Columns include `verification_status`, `contract_type`, `rarity_score`, `last_sale_price`, `last_sale_date`, and `total_supply`.

At this scale, a few habits matter. First, preview with `df.head()` and check `df.shape` before pivoting, because a pivot on high-cardinality or dirty columns can explode into a huge, unreadable table. Second, confirm the dtypes of the columns you will aggregate — `last_sale_price` and `rarity_score` must be numeric for `sum` to be meaningful, and a stray non-numeric row could force the whole column to `object`. Third, decide up front which columns belong on the axes and which are the measure. Here `verification_status` and `contract_type` are low-cardinality categories (good for index/columns), while `rarity_score` and `last_sale_price` are the numeric measures.

In [2]:
# code here
df = pd.read_csv('https://tinyurl.com/2pfhncqw')

In [3]:
print(df.shape)
df.head()

(309910, 26)


   Unnamed: 0  rarity_score last_sale_date  last_sale_price  total_supply  \
0           0      2.000000        2022-11         1.266732             1   
1           1     97.077216        2021-08         1.200357          8888   
2           2    139.453086        2022-08         3.180572          8888   
3           3    127.753445        2021-08         3.502910          8888   
4           4    112.315719        2022-09         4.009567          8888   

  creation_date verification_status  n_of_traits contract_type  seller_fees  \
0       2022-04            verified            2           own       1000.0   
1       2021-07            verified            5           own        500.0   
2       2021-07            verified            5           own        500.0   
3       2021-07            verified            5           own        500.0   
4       2021-07            verified            5           own        500.0   

   platform_fees  openrarity_enabled  has_website  has_own_twi

## 1.2 The pivot itself

The answer is one call:
```python
df.pivot_table(index='verification_status', columns='contract_type',
               values=['rarity_score','last_sale_price'], aggfunc='sum')
```
Each row is a verification status, each block of columns a contract type, and the values are the sums. Because `values` is a **list**, Pandas returns columns with two levels — the measure name (`last_sale_price`, `rarity_score`) on top and the contract type (`default`, `own`) beneath. Reading across, the `approved`/`own` cell for `last_sale_price` is about 1684.8 while `approved`/`default` is ~658.9, and the rarity sums are on the order of hundreds of millions (hence the scientific notation in the output).

The default `aggfunc` would have been `mean`, so passing `'sum'` is essential to 'find out the total values'. Adding `fill_value=0` would replace empty combinations, and `margins=True` would append overall totals. This example is a clean demonstration of two things: pivot tables accept a *list* of value columns and produce MultiIndex columns, and the same table is expressible as `df.groupby(['verification_status','contract_type'])[['rarity_score','last_sale_price']].sum().unstack()` if you prefer the explicit route. Choosing sensible categorical axes keeps the result interpretable even at 310,000 source rows.

In [4]:
df.pivot_table(index='verification_status',columns='contract_type', values=['rarity_score','last_sale_price'],aggfunc='sum')

                    last_sale_price                 rarity_score              
contract_type               default            own       default           own
verification_status                                                           
approved                 658.931935    1684.819488  3.973082e+08  3.929607e+06
verified                 370.304092  667394.862655  2.475007e+06  4.165654e+07

# 2. Car Prices — Clean, Pivot, and Plot (Q-2)

The car dataset hides its brand inside the name and corrupts some years. Both must be fixed before the pivot means anything.

### `Question-2:`

You are given a dataset about the cars' price and miles driven of different cars throughout the different years. The link of the dataset: https://tinyurl.com/2r24n45l. Your tasks are
- make a pivot table of the brands (the required brands are given below) from the "Year" 2018 to the year 2022 in which the "Price" is shown as average values and "Miles" are in median values. In this pivote table, every row represents a particular brand and each column represents either average "Price" of a partucular year or median values of "Miles" of a particular year.
- At the end plot a kde chart for the "Price" and "Miles" by using your pivot table.

There are some challenges to solve this task:
- There is no column that is represented to the brand name of the car. You have to find out on your own.
- Some values of "Years" column are misleading. You have to reset this column too that every value should tell a valid meaning.

## 2.1 Loading and deriving the brand

`pd.read_csv('https://tinyurl.com/2r24n45l')` loads 22,000 rows with columns `Name`, `Year`, `Miles`, and `Price`. `df.head()` shows names like `Chevrolet Trax`, `GMC Terrain`, and `Jeep Wrangler`, and — the challenge — confirms there is **no brand column**.

The fix extracts the first word of `Name`: `df['Brand'] = df['Name'].str.split(' ').str.get(0)`. `str.split(' ')` produces a list of words per name and `.str.get(0)` keeps the first, giving `Chevrolet`, `GMC`, `Jeep`, and so on. This works because these names follow the convention *Brand Model*; a name like `Land Rover Range Rover` would misassign the brand, so in production you would either maintain a brand list or a smarter regex. Note also the one-row peek showing `BMW X` with `Year 20173` — the corrupted year that the next step repairs. Deriving a missing categorical feature from a composite text column is a very common real-world task, and `.str.split().str.get(n)` is the standard first tool for it.

In [40]:
# code here
df = pd.read_csv('https://tinyurl.com/2r24n45l')
df.head()

             Name   Year  Miles  Price
0  Chevrolet Trax   2018  41946  16990
1     GMC Terrain   2020  45328  23990
2   Jeep Wrangler   2012  81068  21590
3   Jeep Renegade   2019  35372  21590
4           BMW X  20173  68992  22990

In [6]:
df.shape

(22000, 4)

## 2.2 Repairing the malformed Year values

The row with `Year = 20173` is the 'misleading value' the question warns about — a digit was inserted or concatenated. The chosen fix extracts the first four-digit number from whatever the value is:
```python
df['Year'] = df['Year'].apply(lambda x: int(re.findall(r"\d{4}", str(x))[0]))
```
The regex `\d{4}` finds a run of exactly four digits, and `str(x)` first stringifies the value so the same code works whether the entry arrived as an integer, a float, or text. The standalone demonstration cell `int(re.findall(r"\d{4}", str(2015))[0])` returns `2015`, showing the mechanism in isolation.

After running this, `df['Year']` is a clean integer column where every value is a plausible model year; `df.head()` shows `2018`, `2020`, `2012`, `2019`, and `2015` for the first rows. Two cautions: `re.findall(...)[0]` raises `IndexError` if no four-digit run exists, so `errors`-style guarding (`[0] if matches else np.nan`) is safer on messy production data; and always re-validate with `df['Year'].value_counts()` or a range check after such a transformation, since this kind of silent repair is easy to get subtly wrong.

In [41]:
df['Brand'] = df['Name'].str.split(" ").str.get(0)

In [31]:
df.head()

             Name   Year  Miles  Price      Brand
0  Chevrolet Trax   2018  41946  16990  Chevrolet
1     GMC Terrain   2020  45328  23990        GMC
2   Jeep Wrangler   2012  81068  21590       Jeep
3   Jeep Renegade   2019  35372  21590       Jeep
4           BMW X  20173  68992  22990        BMW

In [36]:
import re
int(re.findall(r"\d{4}", str(2015))[0])

2015

In [42]:
df['Year'] = df['Year'].apply(lambda x:int(re.findall(r"\d{4}", str(x))[0]))

## 2.3 The pivot and KDE plots

With `Brand` and a clean `Year` in place, the pivot restricts to the years 2018–2022 and applies two different aggregations:
```python
pt = df.query('Year >= 2018 and Year <= 2022').pivot_table(
    index='Brand', columns='Year', values=['Price','Miles'],
    aggfunc={'Price':'mean','Miles':'median'})
```
The dict form of `aggfunc` is the key detail: `Price` is averaged (its central tendency) while `Miles` is medianed (robust to the long tail of very high-mileage cars). The result has MultiIndex columns — measure on top, year below — so every row is a brand and each column either a mean price or median mileage for a given year. `query('Year >= 2018 and Year <= 2022')` filters using an expression string, a readable alternative to a chained boolean mask.

The plots then slice each measure from the pivot: `pt['Miles'].plot(kind='kde')` and `pt['Price'].plot(kind='kde')`. A **kernel density estimate** smooths the distribution of those yearly values across brands, showing shape rather than raw counts — useful for spotting whether prices cluster or spread. Remember `kde` requires enough distinct numeric values (ideally more than a handful), and any `NaN` cells from missing brand/year combinations can distort the curve, so inspect `pt` for nulls before trusting the plot.

In [46]:
pt = df.query('Year >= 2018 and Year <= 2022').pivot_table(index='Brand',columns='Year',values=['Price','Miles'],aggfunc={'Price':'mean','Miles':'median'})

In [49]:
pt['Miles'].plot(kind='kde')

<Figure size 432x288 with 1 Axes>

In [50]:
pt['Price'].plot(kind='kde')

<Figure size 432x288 with 1 Axes>

# 3. India's Daily Power Generation (Q-3)

A large operational dataset — 345,273 rows — where generation must be rebalanced, months extracted, stations ranked, and a pivot plotted.

### `Question-3:`

You are given a dataset of **Daily Power Generation in India** of regional wise of all Power Stations. Link of the dataset: https://tinyurl.com/2nq6kugt

Task
- In this dataset, there are many columns. Two of them are `Actual(MU)` and `Excess(+) / Shortfall (-)`. `Actual(MU)` represents the actual power generation. `Excess(+) / Shortfall (-)` tells is that generated power is excess or shortfall for that particular day of a power station. You have to find out what should be the actual power generation required for that day by the power stations that there would not be any excess or shorfall power. For the result, make a new column.
- Find out the month of the day and store it as a new column.
- Find out top 10 frequently appeared power stations in this dataframe.
- Next create a pivot table of which every row represents a power station and every column represents a month in a order. Like January, February, March...
- It is hard to get inside if you look through the only pivot table. So plot the pivot table.

## 3.1 Loading and inspecting

`pd.read_csv('https://tinyurl.com/2nq6kugt')` loads the daily power-generation data; `df.head()` shows `Dates`, `Power Station`, `Monitored Cap.(MW)`, and further operational columns, and `df.shape` confirms **(345273, 11)**. Each row is one station's generation on one date, so the two columns the task cares about — `Actual(MU)` (actual generation in million units) and `Excess(+) / Shortfall (-)` (the gap between actual and required) — sit alongside regional and capacity information.

At this size, always confirm the numeric dtypes of the measure columns before arithmetic. The `Dates` column arrives as text (`'2017-09-01'`), which the next step parses. The grain — one station per date — means date-based grouping (month) and station-based grouping (top-10, pivot) both make sense, and the column names contain parentheses and symbols, so reference them with exact strings (`df['Actual(MU)']`, `df['Excess(+) / Shortfall (-)']`) or rename them early for convenience. Understanding the grain and the dtypes up front prevents silent errors in the derivations that follow.

In [51]:
# code here
df = pd.read_csv('https://tinyurl.com/2nq6kugt')
df.head()

        Dates      Power Station  Monitored Cap.(MW)  \
0  2017-09-01              Delhi              2235.4   
1  2017-09-01            Haryana              2720.0   
2  2017-09-01   Himachal Pradesh              3378.0   
3  2017-09-01  Jammu and Kashmir              1285.0   
4  2017-09-01             Punjab              3826.3   

   Total Cap. Under Maintenace (MW)  Planned Maintanence (MW)  \
0                            135.00                      0.00   
1                           2470.00                      0.00   
2                            379.00                      0.00   
3                            150.00                      0.00   
4                           2697.65                     77.65   

   Forced Maintanence(MW)  Other Reasons (MW)  Programme or Expected(MU)  \
0                   135.0                   0                         13   
1                  2470.0                   0                         28   
2                   231.0                   

In [52]:
df.shape

(345273, 11)

## 3.2 The required-generation column

The goal is the generation that would have produced neither surplus nor shortfall. Since `Excess(+) / Shortfall (-)` is the signed difference between actual and required, the requirement is:
```python
df['required'] = df['Actual(MU)'] - df['Excess(+) / Shortfall (-)']
```
On a surplus day the excess is positive, so subtracting it *lowers* the requirement below actual; on a shortfall day it is negative, so subtracting a negative *raises* the requirement above actual. One expression handles both signs correctly — the elegance of keeping the sign rather than taking absolute values. Validate on a couple of rows: a `+5` excess should yield a `required` five below `Actual`, and a `-5` shortfall five above.

This is a simple but important modelling habit: when a dataset gives you one quantity as a *deviation* from a target, the target is recovered by subtracting the deviation. It is the same algebra behind `shift`-based diffs and residual analysis. Watch for `NaN` in either source column (the result inherits the missingness) and for units — both columns must be in the same unit (MU) for the subtraction to be valid, which they are here.

In [53]:
df['required'] = df['Actual(MU)'] - df['Excess(+) / Shortfall (-)']
df.head()

        Dates      Power Station  Monitored Cap.(MW)  \
0  2017-09-01              Delhi              2235.4   
1  2017-09-01            Haryana              2720.0   
2  2017-09-01   Himachal Pradesh              3378.0   
3  2017-09-01  Jammu and Kashmir              1285.0   
4  2017-09-01             Punjab              3826.3   

   Total Cap. Under Maintenace (MW)  Planned Maintanence (MW)  \
0                            135.00                      0.00   
1                           2470.00                      0.00   
2                            379.00                      0.00   
3                            150.00                      0.00   
4                           2697.65                     77.65   

   Forced Maintanence(MW)  Other Reasons (MW)  Programme or Expected(MU)  \
0                   135.0                   0                         13   
1                  2470.0                   0                         28   
2                   231.0                   

In [56]:
df['month'] = pd.to_datetime(df['Dates']).dt.month_name()
df.head()

        Dates      Power Station  Monitored Cap.(MW)  \
0  2017-09-01              Delhi              2235.4   
1  2017-09-01            Haryana              2720.0   
2  2017-09-01   Himachal Pradesh              3378.0   
3  2017-09-01  Jammu and Kashmir              1285.0   
4  2017-09-01             Punjab              3826.3   

   Total Cap. Under Maintenace (MW)  Planned Maintanence (MW)  \
0                            135.00                      0.00   
1                           2470.00                      0.00   
2                            379.00                      0.00   
3                            150.00                      0.00   
4                           2697.65                     77.65   

   Forced Maintanence(MW)  Other Reasons (MW)  Programme or Expected(MU)  \
0                   135.0                   0                         13   
1                  2470.0                   0                         28   
2                   231.0                   

In [60]:
top_10 = df['Power Station'].value_counts().head(10).index.to_list()

## 3.3 Month extraction, top stations, and the pivot plot

**Month.** `df['month'] = pd.to_datetime(df['Dates']).dt.month_name()` parses the text dates and extracts the month *name* (January, February, ...). Strings sort alphabetically, so January–December will plot out of chronological order unless you either map them to a `pd.Categorical` with the twelve names in order or pivot on the numeric month and relabel.

**Top 10 stations.** `df['Power Station'].value_counts().head(10).index.to_list()` ranks stations by number of appearances and returns the ten busiest names as a list.

**The pivot.** `df[df['Power Station'].isin(top_10)].pivot_table(index='month', columns='Power Station', values='required', aggfunc='sum')` restricts to those ten stations, then builds a month × station grid of summed required generation. `isin(top_10)` is the readable way to filter a column to a list of allowed values.

**The plot.** `pt.plot(kind='line')` draws one line per station across the months — exactly the visualisation the question asks for, since 'it is hard to get inside' a wide numeric pivot. A line chart exposes seasonality (higher summer demand) and relative station scale at a glance. With ten overlapping lines the legend may crowd; `kind='area'`, a stacked bar, or normalising each row would clarify the comparison further.

In [65]:
pt = df[df['Power Station'].isin(top_10)].pivot_table(index='month',columns='Power Station',values='required',aggfunc='sum')

In [66]:
pt.plot(kind='line')

<Figure size 432x288 with 1 Axes>

# 4. Parsing question-answer.csv (Q-4, Q-5)

A file that is really a text dump of numbered questions and answers must be turned into a tidy two-column table, then filtered.

###`Q-4` You are given a file `question-answer.csv`. Your task is to make a dataframe from it with two columns - `question` and `answers`.

* Questions in the file start from `Q<number>`; E.g.  `Q1` denotes question no. `1`
* Answers starts from `Ans<number>`; Eg. `Ans1` denotes answer of question no `1`
* MAke sure you look at columns name carefully

CSV File - "https://drive.google.com/file/d/10rmV3XrVtzpDTtYZF3UtCdcU0ajBJjGY/view?usp=share_link"

## 4.1 Reading the raw file

`pd.read_csv('/content/question-answer.csv')` reads the file in whichever default shape fits, and the preview shows it has **two columns**: a junk `Unnamed: 0` index column and a data column literally named `'2'`. The rows alternate between questions (`Q1-What is Data Science?`) and answers (`Ans1-Data Science combines statistics...`), so `df.shape` reports (20, 2) — ten Q/A pairs. The key observations: the data column's name is the string `'2'` (not a descriptive name), and the question/answer marker is embedded as a **prefix** inside the text, separated from the content by a hyphen. This is the classic 'unstructured text in a CSV' situation — the parsing work happens after reading, and the column name must be quoted exactly (`df['2']`) because it is not a valid identifier.

In [75]:
# code here
df = pd.read_csv("/content/question-answer.csv")
df

    Unnamed: 0                                                  2
0            0                           Q1-What is Data Science?
1            1  Ans1-Data Science combines statistics, maths, ...
2            2             Q2-2. How is logistic regression done?
3            3  Ans2-Logistic regression measures the relation...
4            4         Q3-How do you build a random forest model?
5            5  Ans3-A random forest is built up of a number o...
6            6       Q4-How can you avoid overfitting your model?
7            7  Ans4-Overfitting refers to a model that is onl...
8            8  Q5-You are given a data set consisting of vari...
9            9  Ans5-The following are ways to handle missing ...
10          10  Q6-What is bag of words? How we can use it for...
11          11  Ans6-Bag of Words is a representation of text ...
12          12  Q7-How can we use machine learning for text cl...
13          13  Ans7-Machine learning classification algorithm...
14        

In [70]:
df.shape

(20, 2)

## 4.2 Splitting questions and answers

Questions are those rows where the text **starts with `Q`**:
```python
ques = df['2'][df['2'].str.startswith('Q')].str.split('-').str.get(1).reset_index()
```
The boolean mask `str.startswith('Q')` selects question rows, `.str.split('-')` breaks each into `['Q1', 'What is Data Science?']`, and `.str.get(1)` keeps index 1 — the human-readable question, with the `Q1` prefix discarded. `reset_index()` is crucial: it turns the original row index (0, 2, 4, ...) into a column while creating a fresh 0-based index, so the questions occupy contiguous positions.

Answers are extracted identically with `str.startswith('A')` into `ans` (the marker `Ans1` is dropped by the same `split('-').get(1)`). Note a subtle risk: a *question* that itself contains a hyphen (e.g. `Q2-2. How is logistic regression done?` has no hyphen, but many do) would be split into more than two parts, and `.str.get(1)` would truncate it. A regex `str.split('-', n=1).str.get(1)` (limit to one split) handles hyphens in the content correctly. Both `ques` and `ans` are now clean one-column frames aligned by their new 0-based indexes.

In [83]:
ques = df['2'][df['2'].str.startswith('Q')].str.split("-").str.get(1).reset_index()

In [85]:
ques

   index                                                  2
0      0                              What is Data Science?
1      2                2. How is logistic regression done?
2      4            How do you build a random forest model?
3      6          How can you avoid overfitting your model?
4      8  You are given a data set consisting of variabl...
5     10  What is bag of words? How we can use it for te...
6     12  How can we use machine learning for text class...
7     14                             What is a time series?
8     16  How is time series different from the usual re...
9     18  What are the different types of joins? What ar...

In [86]:
ans = df['2'][df['2'].str.startswith('A')].str.split("-").str.get(1).reset_index()

## 4.3 Joining the two halves

With `ques` and `ans` each having a fresh contiguous index, they are combined by that index:
```python
result = ques.merge(ans, left_index=True, right_index=True) \
    .drop(columns=['index_x','index_y']) \
    .rename(columns={'2_x':'Question', '2_y':'Answer'})
```
`merge(..., left_index=True, right_index=True)` performs an **index-based** join rather than a column join — appropriate because there is no shared natural key, only the position that pairs question *n* with answer *n*. The leftover `index_x`/`index_y` columns (from the earlier `reset_index` calls) are dropped, and the two data columns renamed to `Question` and `Answer`.

The result is a tidy two-column DataFrame with ten rows, e.g. `What is Data Science?` paired with `Data Science combines statistics, maths, ...`. Two caveats: this position-based join silently assumes every question has a matching answer in order — if the file were missing an answer the alignment would shift, so a robust parser would extract the *number* from each marker (`Q1`/`Ans1`) and merge on that number instead. And the `_x`/`_y` suffixes appear automatically when both frames share a column name, which is why the rename is necessary.

In [92]:
result = ques.merge(ans, left_index=True, right_index=True).drop(columns=['index_x','index_y']).rename(columns={'2_x':'Question','2_y':'Answer'})

In [93]:
result

                                            Question  \
0                              What is Data Science?   
1                2. How is logistic regression done?   
2            How do you build a random forest model?   
3          How can you avoid overfitting your model?   
4  You are given a data set consisting of variabl...   
5  What is bag of words? How we can use it for te...   
6  How can we use machine learning for text class...   
7                             What is a time series?   
8  How is time series different from the usual re...   
9  What are the different types of joins? What ar...   

                                              Answer  
0  Data Science combines statistics, maths, speci...  
1  Logistic regression measures the relationship ...  
2  A random forest is built up of a number of dec...  
3  Overfitting refers to a model that is only set...  
4  The following are ways to handle missing data ...  
5  Bag of Words is a representation of text that ... 

# Task Q-5 — Questions Without a Question Mark

**Goal:** Print the question and answer of entries whose question contains no `?`.

###`Q-5`: Print Question and answer of those questions which does not contains any question mark (`?`).


## 5.1 Approach — escaped regex on text

The filter is `result[~result['Question'].str.contains('\?')]`. Two details make it correct. First, `?` is a **regular-expression metacharacter** meaning 'optional previous token', so it must be escaped as `\?` to match a literal question mark — otherwise the pattern would match far too much. Second, the tilde `~` negates the boolean mask, keeping the rows where no `?` was found.

The executed output is an **empty DataFrame** — every question in this file is phrased as a question ending in `?`, so the negation selects nothing. That empty result is itself informative: it confirms the parser captured the `?` correctly (had the split truncated it, questions would appear to lack one and the filter would return rows). On a real FAQ dataset, this filter is a useful way to find *statement-style* entries that were mislabelled as questions or missing punctuation. The pattern generalises: `str.contains` with `regex=False` for plain substrings (faster), escaped metacharacters when a regex is needed, and `~mask` for 'does not contain'.

In [98]:
# code here
result[~result['Question'].str.contains('\?')]

Empty DataFrame
Columns: [Question, Answer]
Index: []

# 5. LOG and EMPLOYEE (Q6–Q10)

Two tables joined on mismatched keys — `log_file.emp_id` and `employee.EMPLOYEE_ID` — to answer activity questions by month, weekday, and weekend.

###`Q 6-10` LOG and EMPLOYEE 
6. Show `activity` details month wise. Show count for each `activity` 
7. Find employee who did most `activity` in January month.
8. Employee who have worked most no of times on Weekends.
9. Which activity is logged most on buisness days.
10. Week Days wise activity table.
```
log_file = pd.read_csv("https://docs.google.com/spreadsheets/d/e/2PACX-1vT-OMrmmNdOTM-B5f5F1EpCutMVG230UZiLvqlsg0NIKUKR3yrqiI2r1pEX-LvSEk-3WwySPYtvbBC-/pub?gid=1937029224&single=true&output=csv")
employee = pd.read_csv("https://docs.google.com/spreadsheets/d/e/2PACX-1vQ5AuFqRjSZVBKm5zyDxquX6utubq1DJKkYDI70vjeidAnyAu70KMSYpMYzeVSNVTeUIJBpfF6jU5E6/pub?gid=798824749&single=true&output=csv")

```

Note(for common field):- Employee file has `EMPLOYEE_ID` and LOG file has `emp_id`

## 5.0 Loading the two files

`pd.read_csv` loads `log_file` and `employee` from their published URLs. The log preview shows `Log_ID`, `dt`, `dept`, `emp_id`, `activity`, and `eqpt` — one row per logged activity event, with activities like `Cleaning`, `Inspection`, `Checking`, and `Repair`. The employee file carries `EMPLOYEE_ID`, `FIRST_NAME`, `LAST_NAME`, and `EMAIL`. The stated join key is the mismatch `emp_id` (log) versus `EMPLOYEE_ID` (employee); normalising that is the first task, whether by renaming or by specifying `left_on`/`right_on` at merge time. The `dt` column arrives as text and must be parsed before any date extraction, and the activity spellings — including the dataset's own typo `'Incpection'` — must be matched exactly when filtering, so inspect `value_counts()` on `activity` first to see the real category set.

In [99]:
# code here
log_file = pd.read_csv("https://docs.google.com/spreadsheets/d/e/2PACX-1vT-OMrmmNdOTM-B5f5F1EpCutMVG230UZiLvqlsg0NIKUKR3yrqiI2r1pEX-LvSEk-3WwySPYtvbBC-/pub?gid=1937029224&single=true&output=csv")
employee = pd.read_csv("https://docs.google.com/spreadsheets/d/e/2PACX-1vQ5AuFqRjSZVBKm5zyDxquX6utubq1DJKkYDI70vjeidAnyAu70KMSYpMYzeVSNVTeUIJBpfF6jU5E6/pub?gid=798824749&single=true&output=csv")

In [104]:
log_file.head()

   Log_ID          dt   dept  emp_id    activity       eqpt
0    1000  2022-01-07  Dept6     115    Cleaning        UPS
1    1001  2022-01-08  Dept4     105    Cleaning  Generator
2    1002  2022-01-09  Dept3     112  Inspection        CPU
3    1003  2022-01-10  Dept4     120  Inspection        UPS
4    1004  2022-01-11  Dept3     138    Checking    Battery

## 5.1 Parsing the date and deriving time fields

`log_file['dt'] = pd.to_datetime(log_file['dt'])` converts the date column to `datetime64`, which is the prerequisite for every time-based question. From it you can then pull month and weekday using the `.dt` accessor:
```python
log_file['month'] = log_file['dt'].dt.month_name()
log_file['day'] = log_file['dt'].dt.day_name()
```
`month_name()` returns `'January'`, `'February'`, etc., and `day_name()` returns `'Monday'`...`'Sunday'`. For weekend logic you can use `log_file['dt'].dt.day_of_week`, where Monday is 0 and Sunday is 6 — so `> 4` identifies Saturday and Sunday (weekends) and `<= 4` identifies business days. Deriving these fields once, rather than recomputing per question, is both cleaner and faster. A production habit: verify the parse succeeded by checking the dtype and that no rows became `NaT` (`log_file['dt'].isna().sum()`), since an unparseable date would silently drop a row from every later aggregation.

In [107]:
log_file['dt'] = pd.to_datetime(log_file['dt'])

In [111]:
log_file['month'] = log_file['dt'].dt.month_name()

## 5.2 Month-wise activity counts (Q6)

`log_file.pivot_table(index='month', columns='activity', aggfunc='count')['Log_ID']` creates a month × activity grid where each cell counts the matching `Log_ID`s. Because `aggfunc='count'` must know *what* to count, it counts every column and the `['Log_ID']` selection keeps just the primary-key column, giving a clean count table. The output shows, for April, Checking 5, Cleaning 11, Inspection 9, Repair 5, and so on across the months. This is the classic 'how many events of each type per month' cross-tab, and it is equivalent to `pd.crosstab(log_file['month'], log_file['activity'])` — `crosstab` being the purpose-built (and slightly more readable) tool for count tables. Two refinements: `fill_value=0` makes empty month/activity combinations show as zero rather than `NaN`, and reindexing the rows to calendar order (via `pd.Categorical` or a sorted key) keeps the months chronological instead of alphabetical.

In [114]:
log_file.pivot_table(index='month',columns='activity',aggfunc='count')['Log_ID']

activity   Checking  Cleaning  Inspection  Repair
month                                            
April             5        11           9       5
August            8        10           9       4
December          6         7          12       6
February          7         7           4      10
January           7        10           9       5
July              6         6          14       5
June              5         7           8      10
March             8         6           5      12
May               5         8           7      11
November          8         9           6       7
October          13         4           5       9
September         8         3          12       7

## 5.3 Most active employee in January (Q7)

`log_file.query('month == "January"')['emp_id'].value_counts().head(1)` filters to January using a query string, counts log entries per employee, and takes the top one — employee **122 with 3 activities**. The chain is the standard 'filter → value_counts → top' idiom: `query` keeps January rows, `value_counts()` ranks employees by number of logs, and `.head(1)` isolates the busiest. `query` is a readable alternative to `log_file[log_file['month'] == 'January']`, and it supports `and`/`or`/`in` syntax for multi-condition filters. To show the employee's *name* rather than their id, you would merge the id back to the employee table on `emp_id`/`EMPLOYEE_ID`. Also be aware that `value_counts` excludes `NaN` keys by default; if some logs lacked an `emp_id` they would be dropped silently — check `isna()` if that is a concern.

In [117]:
log_file.query('month == "January"')['emp_id'].value_counts().head(1)

122    3
Name: emp_id, dtype: int64

## 5.4 Most weekend work and top business-day activity (Q8, Q9)

Two complementary filters use the weekday number.

**Weekends (Q8):** `log_file[log_file['dt'].dt.day_of_week > 4]['emp_id'].value_counts().head(1)` — `day_of_week > 4` selects Saturday (5) and Sunday (6), then ranks employees by how many weekend logs they have. The answer is employee **144 with 6** weekend activities. This identifies who carries the weekend operational load, valuable for fairness and rostering reviews.

**Business days (Q9):** `log_file[log_file['dt'].dt.day_of_week <= 4]['activity'].value_counts().head(1)` — `<= 4` keeps Monday–Friday, and counting the `activity` column finds the most frequent one: **Inspection with 77**. 

Both use the same shape: mask by a weekday condition, then `value_counts()` the interesting column. Note the asymmetry — Q8 counts employees (who did the most) while Q9 counts activities (what was done most) — so choosing the right column to `value_counts` is the whole decision. Also be careful with `day_of_week` conventions: if you had used `weekday` versus `day_of_week` or assumed Sunday = 0, the weekend mask would be off by a day; always confirm the numbering from the docs (`day_of_week`: Monday=0 … Sunday=6).

In [124]:
log_file[log_file['dt'].dt.day_of_week > 4]['emp_id'].value_counts().head(1)

144    6
Name: emp_id, dtype: int64

In [125]:
log_file[log_file['dt'].dt.day_of_week <= 4]['activity'].value_counts().head(1)

Inspection    77
Name: activity, dtype: int64

## 5.5 Weekday-wise activity table (Q10)

The final log question builds a full weekday × activity matrix. First `log_file['day'] = log_file['dt'].dt.day_name()` adds the weekday name column, then:
```python
log_file.pivot_table(index='day', columns='activity', aggfunc='count')['Log_ID']
```
produces a grid with one row per weekday and one column per activity, each cell a count. The output shows Friday 8/16/20/9, Monday 10/14/13/15, Saturday 18/11/8/15, Sunday 13/13/15/11 across Checking/Cleaning/Inspection/Repair — enough to see that Checking peaks on weekends while Inspection is relatively steady. As with Q6, `pd.crosstab(log_file['day'], log_file['activity'])` is the direct equivalent, and both benefit from `fill_value=0`.

One presentational improvement worth making: the day names sort alphabetically, so the table appears as Friday, Monday, Saturday, Sunday, ... To display Monday→Sunday, reindex the rows against an ordered day list or convert `day` to a `pd.Categorical` with the seven names as categories *before* pivoting. The same ordering fix applies to months. With the table correctly ordered and zero-filled, it becomes a legible heatmap of the week's operational rhythm — the perfect summary of questions Q6–Q10.

In [126]:
log_file['day'] = log_file['dt'].dt.day_name()
log_file.head()

   Log_ID         dt   dept  emp_id    activity       eqpt    month       day
0    1000 2022-01-07  Dept6     115    Cleaning        UPS  January    Friday
1    1001 2022-01-08  Dept4     105    Cleaning  Generator  January  Saturday
2    1002 2022-01-09  Dept3     112  Inspection        CPU  January    Sunday
3    1003 2022-01-10  Dept4     120  Inspection        UPS  January    Monday
4    1004 2022-01-11  Dept3     138    Checking    Battery  January   Tuesday

In [127]:
log_file.pivot_table(index='day',columns='activity',aggfunc='count')['Log_ID']

activity   Checking  Cleaning  Inspection  Repair
day                                              
Friday            8        16          20       9
Monday           10        14          13      15
Saturday         18        11           8      15
Sunday           13        13          15      11
Thursday          8        16          16      12
Tuesday          15         9          15      13
Wednesday        14         9          13      16

# 6. Cleaning the Employee File (Q-11)

The final question repairs the employee table itself: recover missing first names, complete the email domain, and filter names by activity.

### `Q-11`:

1. There are missing value in first name of employee. Fill it using email and last name field.

> E.g- `email` -> JMURMAN and `last_name` -> 'Urman' -> so make it's first name as 'JM', Sort of (Email- lastname). 

> Email is constructed from initials of first name concate with lastname. Your Task is to fill first name initials in missing data.

2. You can see email field don't have any domain name. Change this to full email addreess with domain as 'campusx.com'.
E.g- Email field -> `JMURMAN`  result -> `JMURMAN@CAMPUSX.COM`

3. Show Full Name of all the employees whose name starts with 'A' and has done any of these activity ['Incpection', 'Cleaning', 'Checking]

In [147]:
# code here
employee = pd.read_csv("https://docs.google.com/spreadsheets/d/e/2PACX-1vQ5AuFqRjSZVBKm5zyDxquX6utubq1DJKkYDI70vjeidAnyAu70KMSYpMYzeVSNVTeUIJBpfF6jU5E6/pub?gid=798824749&single=true&output=csv")

## 6.1 Reconstructing missing first names

The dataset's missing `FIRST_NAME` values are recoverable by a stated rule: the email stem is the *initials of the first name concatenated with the last name*. So for `EMAIL = 'JMURMAN'` and `LAST_NAME = 'Urman'`, the missing first name initials are everything before the last name's characters — `'JM'`.
```python
fill_values = employee[employee['FIRST_NAME'].isnull()].apply(
    lambda x: x['EMAIL'][0] + x['LAST_NAME'][0], axis=1)
employee['FIRST_NAME'].fillna(fill_values, inplace=True)
```
The row-wise `apply(axis=1)` operates on each missing-name row, taking the first letter of the email (`x['EMAIL'][0]`) and the first letter of the last name (`x['LAST_NAME'][0]`) and concatenating them. The output shows the three fills: row 12 → `'JU'`, row 21 → `'AF'`, row 32 → `'TO'`. `Series.fillna` accepts a **Series** aligned by index, so the fills land on exactly the null rows.

A more robust rule would strip the *full* uppercased last name from the email (`.replace(x['LAST_NAME'].upper(), '')`) rather than just taking the first two letters, which correctly handles multi-letter initials such as `'JM'` from `JMURMAN` minus `URMAN`. This exercise is a good example of **rule-based imputation**: when a column is a deterministic function of others, you can rebuild it instead of dropping the row.

In [157]:
fill_values = employee[employee['FIRST_NAME'].isnull()].apply(lambda x:x['EMAIL'][0] + x['LAST_NAME'][0],axis=1)

In [158]:
fill_values

12    JU
21    AF
32    TO
dtype: object

In [159]:
employee['FIRST_NAME'].fillna(fill_values,inplace=True)

In [160]:
employee

    EMPLOYEE_ID FIRST_NAME    LAST_NAME      EMAIL
0           100     Steven         King      SKING
1           101      Neena      Kochhar   NKOCHHAR
2           102        Lex      De Haan    LDEHAAN
3           103  Alexander       Hunold    AHUNOLD
4           104      Bruce        Ernst     BERNST
5           105      David       Austin    DAUSTIN
6           106      Valli    Pataballa   VPATABAL
7           107      Diana      Lorentz   DLORENTZ
8           108      Nancy    Greenberg   NGREENBE
9           109     Daniel       Faviet    DFAVIET
10          110       John         Chen      JCHEN
11          111     Ismael      Sciarra   ISCIARRA
12          112         JU        Urman    JMURMAN
13          113       Luis         Popp      LPOPP
14          114        Den     Raphaely   DRAPHEAL
15          115  Alexander         Khoo      AKHOO
16          116     Shelli        Baida     SBAIDA
17          117      Sigal       Tobias    STOBIAS
18          118        Guy     

## 6.2 Completing the email domain

`employee['EMAIL'] = employee['EMAIL'] + '@campusx.com'` appends the domain to every bare email stem, turning `'SKING'` into `'SKING@campusx.com'` and so on. Because it is a simple string concatenation on a Series, it is vectorised and fast.

Two production cautions. First, running this twice would double the domain (`'SKING@campusx.com@campusx.com'`), so an idempotent version guards against rows already containing `@`: `employee.loc[~employee['EMAIL'].str.contains('@'), 'EMAIL'] += '@campusx.com'`. Second, if the desired output format is uppercase (`'JMURMAN@CAMPUSX.COM'` as the question's example shows), you must uppercase both sides: `employee['EMAIL'].str.upper() + '@CAMPUSX.COM'`, since the current stems are already uppercase but the appended domain would otherwise be lowercase. Validate the result by checking `.str.contains('@').all()` and eyeballing `employee.head()`, which now shows the four columns `EMPLOYEE_ID`, `FIRST_NAME`, `LAST_NAME`, and `EMAIL` fully populated.

In [163]:
employee['EMAIL'] = employee['EMAIL'] + '@campusx.com'

In [164]:
employee

    EMPLOYEE_ID FIRST_NAME    LAST_NAME                  EMAIL
0           100     Steven         King      SKING@campusx.com
1           101      Neena      Kochhar   NKOCHHAR@campusx.com
2           102        Lex      De Haan    LDEHAAN@campusx.com
3           103  Alexander       Hunold    AHUNOLD@campusx.com
4           104      Bruce        Ernst     BERNST@campusx.com
5           105      David       Austin    DAUSTIN@campusx.com
6           106      Valli    Pataballa   VPATABAL@campusx.com
7           107      Diana      Lorentz   DLORENTZ@campusx.com
8           108      Nancy    Greenberg   NGREENBE@campusx.com
9           109     Daniel       Faviet    DFAVIET@campusx.com
10          110       John         Chen      JCHEN@campusx.com
11          111     Ismael      Sciarra   ISCIARRA@campusx.com
12          112         JU        Urman    JMURMAN@campusx.com
13          113       Luis         Popp      LPOPP@campusx.com
14          114        Den     Raphaely   DRAPHEAL@camp

## 6.3 Filtering by first-name initial and activity

The last task returns the full names of employees whose name starts with 'A' **and** who performed one of `['Incpection', 'Cleaning', 'Checking']`. It joins the two cleaned tables and applies two masks.

First filter the log to the relevant activities: `temp_df = log_file[log_file['activity'].isin(['Incpection', 'Cleaning', 'Checking'])]`. Note the deliberate typo `'Incpection'` — it must match the dataset's actual value exactly, or those rows silently vanish; this is why inspecting `activity.value_counts()` first is important.

Then join to the employee table on the mismatched keys: `temp_df.merge(employee, right_on='EMPLOYEE_ID', left_on='emp_id')`. The `left_on`/`right_on` form exists precisely for keys with different names, producing a combined frame with each log row carrying the employee's names and email.

Finally apply the name mask: `x_df = x_df[x_df['FIRST_NAME'].str.startswith('A')]`, and display the full names with `(x_df['FIRST_NAME'] + ' ' + x_df['LAST_NAME']).unique()`. The result yields `['Alexander Khoo', 'Alexander Hunold']`. `unique()` deduplicates, since an employee with several matching activities appears once per log row. Combine conditions with `&` when both must hold, and remember the join produces one row per matching *log event* — dedupe with `drop_duplicates` on employee if you want one row per person.

In [166]:
temp_df = log_file[log_file['activity'].isin(['Incpection', 'Cleaning', 'Checking'])]

In [169]:
x_df = temp_df.merge(employee,right_on='EMPLOYEE_ID',left_on='emp_id')

In [172]:
x_df

     Log_ID         dt   dept  emp_id  activity       eqpt      month  \
0      1000 2022-01-07  Dept6     115  Cleaning        UPS    January   
1      1038 2022-02-14  Dept2     115  Cleaning        Fan   February   
2      1293 2022-10-27  Dept4     115  Checking        CPU    October   
3      1322 2022-11-25  Dept2     115  Cleaning        Fan   November   
4      1332 2022-12-05  Dept3     115  Checking        UPS   December   
..      ...        ...    ...     ...       ...        ...        ...   
136    1300 2022-11-03  Dept3     128  Cleaning    Battery   November   
137    1364 2023-01-06  Dept3     128  Cleaning         AC    January   
138    1251 2022-09-15  Dept2     104  Checking        CPU  September   
139    1327 2022-11-30  Dept2     104  Cleaning        CPU   November   
140    1334 2022-12-07  Dept4     104  Checking  Generator   December   

           day  EMPLOYEE_ID FIRST_NAME LAST_NAME                EMAIL  
0       Friday          115  Alexander      Khoo   

In [174]:
x_df = x_df[x_df['FIRST_NAME'].str.startswith('A')]

In [178]:
(x_df['FIRST_NAME'] + ' ' + x_df['LAST_NAME']).unique()

array(['Alexander Khoo', 'Alexander Hunold'], dtype=object)

## Summary

This walkthrough solved Task 22's questions end-to-end. Q-1 pivoted a 310k-row NFT dataset, summing two measures across `verification_status` × `contract_type` (producing MultiIndex columns). Q-2 required cleaning before shaping: extracting `Brand` from `Name` with `str.split().str.get(0)` and repairing corrupted years via a `\d{4}` regex, then pivoting to a brand × year table mixing **mean Price** and **median Miles** and plotting KDE distributions. Q-3 derived a `required` generation column from `Actual(MU)` minus the signed excess/shortfall, extracted month names, ranked the top-10 stations, and pivoted month × station into a line chart. Q-4/Q-5 parsed `question-answer.csv` by prefix-splitting on `Q`/`Ans` markers and joined the halves by index, then filtered with an escaped `\?` regex. Q6–Q10 joined log and employee tables across mismatched keys and answered month-, weekday-, weekend-, and business-day activity questions with `pivot_table`, `value_counts`, and `day_of_week` masks. Q-11 restored missing first names by rule, appended the email domain, and combined `.isin` and `.str.startswith` filters. Shape, clean, derive, join, aggregate, visualise — the full loop.